# 05 Artifacts And Checkpoints Walkthrough

This notebook explains what a real nanochat training run leaves behind.

It is intentionally connected to the local `nanochat-artifacts/` folder that was copied back after the RunPod run:

```text
tokenizer/             -> text/token-id conversion
base_checkpoints/      -> pretrained base model
chatsft_checkpoints/   -> chat/SFT model
report/                -> measurements and samples
speedrun.log           -> original training log
sft_retry.log          -> SFT retry log after the hf_transfer issue
identity_conversations.jsonl -> tiny custom SFT identity dataset
```

The goal is to learn what each file is for, what is required for inference, what is required to resume training, and which log lines are worth checking after an expensive run.

In [ ]:
from pathlib import Path
import json
import os
import re
import sys
from collections import defaultdict

repo_root = Path.cwd()
if not ((repo_root / "nanochat").exists() and (repo_root / "pyproject.toml").exists()):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "nanochat").exists() and (candidate / "pyproject.toml").exists():
            repo_root = candidate
            break

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# By default this notebook looks at the local artifacts copied back from RunPod.
# Override this if your artifacts live somewhere else:
#   export NANOCHAT_ARTIFACTS_DIR=/path/to/nanochat-artifacts
artifact_root = Path(os.environ.get("NANOCHAT_ARTIFACTS_DIR", repo_root / "nanochat-artifacts")).expanduser()

def human_size(num_bytes):
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024 or unit == "TB":
            return f"{value:.1f} {unit}"
        value /= 1024

def path_size(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    total = 0
    for item in path.rglob("*"):
        if item.is_file():
            total += item.stat().st_size
    return total

def require_artifacts():
    if not artifact_root.exists():
        raise RuntimeError(
            f"Artifacts folder not found: {artifact_root}\n"
            "Copy the RunPod artifacts into nanochat-artifacts/ or set NANOCHAT_ARTIFACTS_DIR."
        )

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def print_kv(rows, key_width=28):
    for key, value in rows:
        print(f"{key:<{key_width}} {value}")

print(f"repo root:      {repo_root}")
print(f"artifact root:  {artifact_root}")
print(f"exists:         {artifact_root.exists()}")

## Step 1. Artifact Folder Map

After training, nanochat does not save one single magic model file. It saves a small filesystem layout.

For your run, the important top-level folders are:

- `tokenizer/`: the trained tokenizer used by both base and SFT models.
- `base_checkpoints/`: the pretrained model checkpoint.
- `chatsft_checkpoints/`: the supervised-fine-tuned chat checkpoint.
- `report/`: markdown summaries of training/eval metrics and samples.
- `speedrun.log` / `sft_retry.log`: terminal logs, useful for debugging what actually happened.

Notice what is **not** here: the parquet pretraining dataset. That is fine for inference. You need checkpoints and tokenizer, not the raw training data.

In [ ]:
require_artifacts()

top_level = [p for p in artifact_root.iterdir() if not p.name.startswith(".DS_Store")]
for path in sorted(top_level, key=lambda p: p.name):
    kind = "dir " if path.is_dir() else "file"
    print(f"{kind:4} {human_size(path_size(path)):>10}  {path.relative_to(artifact_root)}")

## Step 2. Tokenizer Artifacts

The tokenizer is required for **all** model use.

`tokenizer.pkl` stores the trained BPE encoding: special tokens and `mergeable_ranks`, meaning byte-segment-to-token-id mappings.

`token_bytes.pt` stores the byte length of each token id. nanochat uses this for BPB evaluation: total loss is divided by the number of original text bytes, not by number of tokens.

These files are tiny compared with the model weights, but they are not optional. If you keep a checkpoint without the matching tokenizer, you cannot reliably turn text into the token ids that checkpoint expects.

In [ ]:
require_artifacts()

tokenizer_dir = artifact_root / "tokenizer"
for path in sorted(tokenizer_dir.glob("*")):
    print(f"{path.name:<20} {human_size(path.stat().st_size):>10}")

try:
    import torch
    token_bytes = torch.load(tokenizer_dir / "token_bytes.pt", map_location="cpu")
    print()
    print("token_bytes.pt summary")
    print_kv([
        ("number of token ids", int(token_bytes.numel())),
        ("min bytes per token", int(token_bytes.min().item())),
        ("max bytes per token", int(token_bytes.max().item())),
        ("mean bytes per token", f"{float(token_bytes.float().mean().item()):.2f}"),
    ])
except Exception as exc:
    print(f"Skipped loading token_bytes.pt: {exc}")

## Step 3. Base Checkpoint vs SFT Checkpoint

nanochat saves base and chat/SFT checkpoints separately.

`base_checkpoints/d24/` is the raw pretrained language model. It can complete text, but it is not yet tuned to behave like an assistant.

`chatsft_checkpoints/d24/` starts from the base model and then trains on chat/instruction data. This is the checkpoint you normally want for `scripts.chat_web` or `scripts.chat_cli`.

Inside each checkpoint folder:

- `model_*.pt`: model parameters. Required for inference.
- `meta_*.json`: model config and training state. Required to rebuild the same GPT shape.
- `optim_*_rank*.pt`: optimizer state shards. Needed only if you want to resume training. Not needed for inference.

In [ ]:
require_artifacts()

def summarize_checkpoint_folder(label, folder):
    folder = Path(folder)
    model_files = sorted(folder.glob("model_*.pt"))
    meta_files = sorted(folder.glob("meta_*.json"))
    optim_files = sorted(folder.glob("optim_*_rank*.pt"))
    print(label)
    print("-" * len(label))
    print_kv([
        ("folder", folder.relative_to(artifact_root)),
        ("model files", len(model_files)),
        ("metadata files", len(meta_files)),
        ("optimizer shards", len(optim_files)),
        ("folder size", human_size(path_size(folder))),
    ])
    for path in [*model_files, *meta_files, *optim_files[:2]]:
        print(f"  {path.name:<28} {human_size(path.stat().st_size):>10}")
    if len(optim_files) > 2:
        print(f"  ... {len(optim_files) - 2} more optimizer shard(s)")
    print()

summarize_checkpoint_folder("Base checkpoint", artifact_root / "base_checkpoints" / "d24")
summarize_checkpoint_folder("SFT checkpoint", artifact_root / "chatsft_checkpoints" / "d24")

## Step 4. What You Need For Inference vs Resume Training

This is the practical saving rule.

For **chat inference**, keep:

- `tokenizer/tokenizer.pkl`
- `tokenizer/token_bytes.pt`
- `chatsft_checkpoints/d24/model_000483.pt`
- `chatsft_checkpoints/d24/meta_000483.json`

For **base-model inference**, keep the same tokenizer files plus the base `model_*.pt` and `meta_*.json`.

For **resume training**, also keep optimizer shards. With 8 GPUs there are 8 optimizer files because each distributed rank saves its own optimizer state.

In [ ]:
require_artifacts()

inference_needed = [
    artifact_root / "tokenizer" / "tokenizer.pkl",
    artifact_root / "tokenizer" / "token_bytes.pt",
    artifact_root / "chatsft_checkpoints" / "d24" / "model_000483.pt",
    artifact_root / "chatsft_checkpoints" / "d24" / "meta_000483.json",
]
resume_extra = sorted((artifact_root / "chatsft_checkpoints" / "d24").glob("optim_000483_rank*.pt"))

print("Minimum files for SFT/chat inference:")
for path in inference_needed:
    rel = str(path.relative_to(artifact_root))
    print(f"  {rel:<55} {human_size(path.stat().st_size):>10}")

print()
print("Extra files needed to resume SFT training:")
for path in resume_extra:
    rel = str(path.relative_to(artifact_root))
    print(f"  {rel:<55} {human_size(path.stat().st_size):>10}")

## Step 5. Metadata Is The Model Blueprint

The `.pt` file stores tensors, but `meta_*.json` tells nanochat what shape those tensors belong to.

When `checkpoint_manager.load_model(...)` runs, it reads metadata, builds a `GPTConfig`, creates the model object, and then loads the saved tensor values.

This is why metadata is not just a nice-to-have log. It is part of loading the checkpoint correctly.

In [ ]:
require_artifacts()

base_meta = read_json(artifact_root / "base_checkpoints" / "d24" / "meta_005568.json")
sft_meta = read_json(artifact_root / "chatsft_checkpoints" / "d24" / "meta_000483.json")

def summarize_meta(label, meta):
    cfg = meta["model_config"]
    user = meta.get("user_config", {})
    print(label)
    print("-" * len(label))
    print_kv([
        ("step", meta.get("step")),
        ("val_bpb", f"{meta.get('val_bpb'):.4f}" if meta.get("val_bpb") is not None else None),
        ("run", user.get("run")),
        ("sequence_len", cfg.get("sequence_len")),
        ("vocab_size", cfg.get("vocab_size")),
        ("n_layer", cfg.get("n_layer")),
        ("n_head", cfg.get("n_head")),
        ("n_kv_head", cfg.get("n_kv_head")),
        ("n_embd", cfg.get("n_embd")),
        ("window_pattern", cfg.get("window_pattern")),
    ])
    if "dataloader_state_dict" in meta:
        print("dataloader_state_dict:", meta["dataloader_state_dict"])
    if "loop_state" in meta:
        print("loop_state:", meta["loop_state"])
    print()

summarize_meta("Base metadata", base_meta)
summarize_meta("SFT metadata", sft_meta)

## Step 6. The Training Log Tells The Story

The markdown report is clean and summarized. The raw logs show what actually happened in order.

Useful log lines answer questions like:

- Did dataset shards download?
- How many training tokens did base training use?
- How fast were the GPUs?
- Did validation BPB improve?
- Did checkpoints save?
- Did a later phase fail even though an earlier phase succeeded?

In [ ]:
require_artifacts()

def show_matching_lines(path, patterns, limit=80):
    path = Path(path)
    regexes = [re.compile(pattern) for pattern in patterns]
    shown = 0
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line_no, line in enumerate(f, 1):
            if any(regex.search(line) for regex in regexes):
                print(f"{line_no:>5}: {line.rstrip()}")
                shown += 1
                if shown >= limit:
                    print(f"... stopped after {limit} matches")
                    break

speedrun_patterns = [
    r"Downloading \d+ shards",
    r"Done! Downloaded",
    r"Total number of training tokens",
    r"Parameter counts",
    r"^total\s+:",
    r"Step 0?5568 \| Validation bpb",
    r"CORE metric",
    r"Saved model parameters",
    r"Peak memory usage",
    r"Total training time",
]

show_matching_lines(artifact_root / "speedrun.log", speedrun_patterns)

## Step 7. The Failure Was After Base Training

Your first `speedrun.sh` run completed base training, then failed when SFT tried to load SmolTalk from Hugging Face.

The root cause was not model math and not GPU memory. It was this environment mismatch:

```text
HF_HUB_ENABLE_HF_TRANSFER=1, but hf_transfer was not installed
```

That is why the later `sft_retry.log` exists. You retried only the SFT phase after fixing the Hugging Face transfer setting.

In [ ]:
require_artifacts()

failure_patterns = [
    r"hf_transfer",
    r"HF_HUB_ENABLE_HF_TRANSFER",
    r"scripts.chat_sft FAILED",
    r"scripts.chat_eval FAILED",
    r"FileNotFoundError: .*chatsft_checkpoints",
]
show_matching_lines(artifact_root / "speedrun.log", failure_patterns, limit=60)

## Step 8. SFT Retry Log

The retry log is important because it contains the successful SFT run and chat evaluation.

The key checks are:

- Did it load the base checkpoint?
- Did it save `chatsft_checkpoints/d24/model_000483.pt`?
- Did chat evaluation complete?
- Did the final report get regenerated?

In [ ]:
require_artifacts()

sft_patterns = [
    r"Loading model from .*base_checkpoints",
    r"Training mixture",
    r"Number of iterations|Calculated number of iterations",
    r"Step 0?0483 \| Validation bpb",
    r"Minimum validation bpb",
    r"Saved model parameters.*chatsft_checkpoints",
    r"ARC-Easy accuracy",
    r"ARC-Challenge accuracy",
    r"MMLU accuracy",
    r"GSM8K accuracy",
    r"HumanEval accuracy",
    r"SpellingBee accuracy",
    r"Generating report",
]
show_matching_lines(artifact_root / "sft_retry.log", sft_patterns, limit=100)

## Step 9. Report Files Are The Human-Friendly Summary

The report folder gives you the clean summary you want to keep next to the checkpoint.

The most useful files here are:

- `base-model-training.md`: model size, tokens, iterations, time, memory.
- `base-model-evaluation.md`: base BPB, CORE, samples.
- `sft.md`: SFT iterations and validation BPB.
- `chat-evaluation-sft.md`: ARC/MMLU/GSM8K/HumanEval/SpellingBee/ChatCORE.
- `report.md`: combined report.

In [ ]:
require_artifacts()

report_dir = artifact_root / "report"
for path in sorted(report_dir.glob("*.md")):
    print(f"{path.name:<28} {human_size(path.stat().st_size):>10}")

interesting_report_patterns = [
    r"^- Number of parameters",
    r"^- Number of training tokens",
    r"^- Calculated number of iterations",
    r"^- Minimum validation bpb",
    r"^- CORE metric",
    r"^- train bpb",
    r"^- val bpb",
    r"^- ARC-Easy",
    r"^- ARC-Challenge",
    r"^- MMLU",
    r"^- GSM8K",
    r"^- HumanEval",
    r"^- SpellingBee",
    r"^- ChatCORE metric",
]

print("\nImportant report metrics")
print("-" * 24)
for report_name in ["base-model-training.md", "base-model-evaluation.md", "sft.md", "chat-evaluation-sft.md"]:
    path = report_dir / report_name
    print(f"\n{report_name}")
    show_matching_lines(path, interesting_report_patterns, limit=50)

## Step 10. How nanochat Uses These Artifacts

The checkpoint loading path is:

```text
scripts.chat_web
  -> checkpoint_manager.load_model(source="sft")
  -> NANOCHAT_BASE_DIR/chatsft_checkpoints/d24/meta_000483.json
  -> build GPTConfig
  -> NANOCHAT_BASE_DIR/chatsft_checkpoints/d24/model_000483.pt
  -> tokenizer/tokenizer.pkl
```

So if you want to run this trained chat model somewhere else, place the artifact folder somewhere and point nanochat at it with `NANOCHAT_BASE_DIR`.

In [ ]:
require_artifacts()

print("Example: run the copied SFT model with chat_web")
print()
print(f"cd {repo_root}")
print("source .venv/bin/activate")
print(f"export NANOCHAT_BASE_DIR={artifact_root}")
print("python -m scripts.chat_web")
print()
print("For base-model evaluation instead of chat inference, nanochat would read base_checkpoints/ instead of chatsft_checkpoints/.")

## Mental Model

After training, think of the artifacts like this:

```text
tokenizer.pkl       = how text becomes token ids
token_bytes.pt      = how BPB knows byte lengths
model_*.pt          = learned weights
meta_*.json         = architecture and training-state metadata
optim_*_rank*.pt    = resume-training state, not inference state
report/*.md         = human-readable scores and samples
*.log              = raw truth when something fails
```

For inference, the core pair is tokenizer + model checkpoint. For debugging and learning, the most valuable files are metadata, reports, and logs.